# Day 1: Simple RAG Implementation

In this session, we will build a complete RAG pipeline from scratch.
No fancy vector databases yet - just Python lists and Numpy!

## Pipeline Steps:
1.  **Load Document**: Read a text file.
2.  **Chunking**: Split text into smaller pieces.
3.  **Embedding**: Convert chunks into vectors.
4.  **Retrieval**: Find relevant chunks for a query.
5.  **Generation**: Ask LLM to answer using the retrieved context.

In [19]:
import numpy as np
from litellm import completion, embedding
from sklearn.metrics.pairwise import cosine_similarity

# Constants
EMBEDDING_MODEL = "ollama/nomic-embed-text"
LLM_MODEL = "ollama/llama3.2:1b" # Or 'mistral', 'gemma', etc.

## 1. Load Document
Let's create a dummy document about a fictional planet.

In [20]:
document_text = """
The planet Zog is located in the Andromeda galaxy. 
It is known for its purple oceans and green skies.
The inhabitants of Zog are called Zogians. They have three eyes and communicate via telepathy.
The capital city of Zog is Zogopolis, a floating city made of crystal.
Zogians love to eat 'Glarp', a glowing fruit that tastes like spicy mango.
The currency on Zog is 'Stardust', which is harvested from meteor showers.
"""

## 2. Chunking
We'll use simple sentence splitting for this example.

In [21]:
def split_into_sentences(text):
    return [s.strip() for s in text.split('.') if s.strip()]

chunks = split_into_sentences(document_text)
print(f"Created {len(chunks)} chunks:")
for i, c in enumerate(chunks):
    print(f"{i}: {c}")

Created 7 chunks:
0: The planet Zog is located in the Andromeda galaxy
1: It is known for its purple oceans and green skies
2: The inhabitants of Zog are called Zogians
3: They have three eyes and communicate via telepathy
4: The capital city of Zog is Zogopolis, a floating city made of crystal
5: Zogians love to eat 'Glarp', a glowing fruit that tastes like spicy mango
6: The currency on Zog is 'Stardust', which is harvested from meteor showers


## 3. Embedding (Indexing)
We convert each chunk into a vector and store it.

In [22]:
def get_embedding(text):
    try:
        response = embedding(model=EMBEDDING_MODEL, input=[text])
        return response['data'][0]['embedding']
    except Exception as e:
        print(f"Error embedding text: {text[:30]}... {e}")
        return []

# Store embeddings in a list (In-Memory Vector Store)
vector_store = []
print("\nGenerating embeddings...")
for chunk in chunks:
    vec = get_embedding(chunk)
    if vec:
        vector_store.append({"text": chunk, "vector": vec})

print(f"Indexed {len(vector_store)} chunks.")


Generating embeddings...
Indexed 7 chunks.


In [23]:
len(vector_store[0].get("vector"))

768

In [24]:
len(vector_store[1].get("vector"))

768

## 4. Retrieval
Function to find the most relevant chunk.

test --> Embeddding (same Embedding)(X)---- > calculate cosine_similarity wrt all the chunks  --- get the best chunk (highest cosine similarity) --> LLM (prompt with the best chunk) --> answer

In [25]:
def retrieve(query, top_k=1):
    query_vec = get_embedding(query)
    if not query_vec:
        return []

    # Calculate similarities
    scores = []
    for item in vector_store:
        sim = cosine_similarity([query_vec], [item['vector']])[0][0]
        scores.append((sim, item['text']))
    
    # Sort by similarity (descending)
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return scores[:top_k]

# Test Retrieval
query = "What do Zogians eat?"
retrieved_chunks = retrieve(query, top_k=2)

print(f"\nQuery: {query}")
print("Retrieved Context:")
for score, text in retrieved_chunks:
    print(f"- ({score:.4f}) {text}")


Query: What do Zogians eat?
Retrieved Context:
- (0.6327) Zogians love to eat 'Glarp', a glowing fruit that tastes like spicy mango
- (0.5771) The inhabitants of Zog are called Zogians


## 5. Generation (Augmented)
Combine context and query to prompt the LLM.

In [26]:
def generate_answer(query, context_chunks):
    context_text = "\n".join([c[1] for c in context_chunks])
    
    prompt = f"""
    You are a helpful assistant. Answer the user's question based ONLY on the provided context.
    If the answer is not in the context, say "I don't know".
    
    Context:
    {context_text}
    
    Question: 
    {query}
    """
    
    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response['choices'][0]['message']['content']

## Full RAG Demo

In [27]:
def run_rag(question):
    print(f"\n--- Asking: {question} ---")
    
    # 1. Retrieve
    # (Since our chunks are small sentences, retrieving 2-3 gives better context)
    context = retrieve(question, top_k=2)
    
    print("Context Found:")
    for _, txt in context:
        print(f" > {txt}")

    # 2. Generate
    answer = generate_answer(question, context)
    print(f"\nAnswer:\n{answer}")

# Let's try some questions!
run_rag("What is the currency of Zog?")
run_rag("Describe the inhabitants of Zog.")
run_rag("How far is Zog from Earth?") # Should answer "I don't know" or similar.


--- Asking: What is the currency of Zog? ---
Context Found:
 > The currency on Zog is 'Stardust', which is harvested from meteor showers
 > The inhabitants of Zog are called Zogians

Answer:
I don't know.

--- Asking: Describe the inhabitants of Zog. ---
Context Found:
 > The inhabitants of Zog are called Zogians
 > The currency on Zog is 'Stardust', which is harvested from meteor showers

Answer:
Our question about the inhabitants of Zog. Here's what I've learned so far:

According to our context, the inhabitants of Zog are called Zogians.

Unfortunately, that's as much information as I have at this point. If you'd like to provide more context or details about Zogians, I'd be happy to help further!

--- Asking: How far is Zog from Earth? ---
Context Found:
 > The inhabitants of Zog are called Zogians
 > The planet Zog is located in the Andromeda galaxy

Answer:
Zog is a planet located approximately 2.5 million light-years away from Earth, in the Andromeda galaxy.
